<a href="https://colab.research.google.com/github/Victor-ro-dev/corn-pest-classifier/blob/main/treinamento_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌽 Treinamento do Modelo de Classificação de Pragas do Milho (MobileNetV2)

Este notebook executa o treinamento do modelo customizado MobileNetV2 utilizando aceleração por GPU no **Google Colab** e exporta o modelo otimizado para **TensorFlow Lite (`.tflite`)**.

---

### 1. Configurando o Repositório no Colab (caso execute do zero)
Se você abriu este notebook diretamente pelo botão do Colab, esta célula clona os arquivos do projeto e entra no diretório:

In [ ]:
import os
if not os.path.exists('src'):
    if not os.path.exists('corn-pest-classifier'):
        !git clone https://github.com/Victor-ro-dev/corn-pest-classifier.git
    if os.path.exists('corn-pest-classifier'):
        %cd corn-pest-classifier
    print('Diretório ativo:', os.getcwd())
else:
    print('Diretório do projeto já configurado:', os.getcwd())

### 2. Verificando o Acelerador de Hardware (GPU)
Certifique-se de que o ambiente de execução está configurado com GPU em: `Ambiente de execução` > `Alterar tipo de ambiente de execução` > `T4 GPU`.

In [ ]:
!nvidia-smi

### 3. Instalação das Dependências do Projeto

In [ ]:
!pip install -q -r requirements.txt

### 4. Coleta de Imagens via GBIF (Opcional)
Você pode baixar fotos de campo de pragas com licenças abertas (CC0 / CC-BY) diretamente pela internet de alta velocidade do Colab:

In [ ]:
# Baixa as imagens de pragas diretamente para dataset/praga (modo binário para o modelo)
!python 01_baixar_gbif.py --max-por-classe 300 --saida dataset/praga --modo binario

### 5. Verificação da Estrutura de Classes do Dataset
Para a classificação binária funcionar, a pasta `dataset/` deve conter:
- `dataset/praga/`: Imagens das pragas do milho (lagartas, vaquinha, etc.).
- `dataset/saudavel/`: Imagens de folhas ou plantas sadias de milho (controle).

In [ ]:
import os
from pathlib import Path

dataset_path = Path('dataset')
if dataset_path.exists():
    classes = [d.name for d in dataset_path.iterdir() if d.is_dir()]
    print(f'Pastas de classes encontradas: {classes}')
    for c in classes:
        imgs = [f for f in (dataset_path / c).glob('*') if f.is_file() and not f.name.endswith('.txt')]
        print(f' - Classe "{c}": {len(imgs)} imagens válidas')
    
    if 'saudavel' in classes and len(list((dataset_path / 'saudavel').glob('*.*'))) <= 1:
        print('\n⚠️ AVISO: A pasta dataset/saudavel está vazia ou contém apenas o README.')
        print('Adicione fotos de folhas sadias em dataset/saudavel antes de iniciar o treinamento!')
else:
    print('Diretório dataset não encontrado. Verifique se o repositório foi clonado corretamente.')

### 6. Visualização de Amostras do Dataset & Data Augmentation

In [ ]:
# Gera as grades visuais do lote e do aumento de dados
!python src/corn_pest_classifier/visualizator.py

# Exibe os gráficos gerados diretamente no Colab
import os
from IPython.display import Image, display

if os.path.exists('preview_batch.png'):
    print('Amostras do lote original de treinamento:')
    display(Image('preview_batch.png'))

if os.path.exists('preview_augmentation.png'):
    print('Efeito das transformações de Data Augmentation:')
    display(Image('preview_augmentation.png'))

### 7. Início do Treinamento e Conversão para TensorFlow Lite (.tflite)

In [ ]:
!python src/corn_pest_classifier/main.py

### 8. Baixar o Modelo Convertido (.tflite) Gerado

In [ ]:
import os
from google.colab import files

modelo_tflite = 'milho_pragas_model.tflite'
if os.path.exists(modelo_tflite):
    print('Iniciando download do modelo .tflite...')
    files.download(modelo_tflite)
else:
    print('Arquivo .tflite ainda não foi gerado.')